# Baseline and small-sample check (rating v0.1)

Answers two design questions with your data:

1. **How strong should the small-sample shrinkage be?** (K = 3 was a guess.) Estimates K from each league's reliability and shows how the players you pick change.
2. **Per-league baseline or one pooled baseline?** Rates the same games both ways and shows whether region style shifts the ratings.

Also lists the `split` labels each league uses (needed to design the Spring / Summer / playoffs filters).

Needs `rating_core.py` and the match-data CSV in the same folder. Press **Run All**.

In [ ]:
# ================= CONFIG =================
FILE = "2026_LoL_esports_match_data_from_OraclesElixir.csv"
LEAGUES = ["LCK", "LPL", "LEC", "LCS", "LCP"]
MIN_GAMES = 15
INCLUDE_PLAYOFFS = False
LOOK_AT = ["Croco", "1xn", "MISSING"]      # players to compare before/after stronger shrinkage

In [ ]:
import numpy as np
import pandas as pd
import rating_core as rc

df = pd.read_csv(FILE, low_memory=False)
cfg = dict(rc.CFG); cfg["MIN_GAMES"] = MIN_GAMES

raw, rated = {}, {}
for lg in LEAGUES:
    p = rc.load_games(df, lg, include_playoffs=INCLUDE_PLAYOFFS)
    if p["gameid"].nunique() < 40:
        print(f"{lg}: only {p['gameid'].nunique()} complete games, skipped"); continue
    raw[lg] = rc.add_raw_stats(p)
    rated[lg] = rc.rate(raw[lg].copy(), cfg)          # each league against its own baseline
    print(f"{lg}: {p['gameid'].nunique()} games")

# split labels per league (to design the period filters)
print("\nSplit labels and number of games, with playoffs included:")
sp = df[(df["league"].isin(raw)) & (df["position"] != "team")]
print(sp.groupby(["league", "split", "playoffs"])["gameid"].nunique().to_string())

In [ ]:
# ---------- 1) how much shrinkage do the data call for? ----------
def estimate_K(p):
    """K = noise / signal for a single game, from the reliability of the full-length average."""
    rels = [rc.split_half(p, "r_overall", how)[1] for how in ("odd/even", "first/second half")]
    R = float(np.nanmean(rels))
    games = p.groupby(["playername", "position"])["gameid"].nunique()
    n = float(games[games >= 10].mean())
    if not (0.05 < R < 0.99):
        return np.nan, R, n
    rho = R / (n - (n - 1) * R)             # reliability of ONE game (Spearman-Brown, inverted)
    return float((1 - rho) / rho), R, n

rows = {lg: dict(zip(["K_estimate", "reliability", "avg_games"], estimate_K(p))) for lg, p in rated.items()}
kt = pd.DataFrame(rows).T
print(kt.round(2).to_string())
K_USE = float(np.clip(np.nanmedian(kt["K_estimate"]), 3, 30))
print(f"\nMedian estimate: K = {K_USE:.1f}   (the current setting is K = 3)")
print("Shrunk rating keeps n / (n + K) of a player's distance from 1.00:")
for n in (15, 25, 50, 80):
    print(f"  {n:3d} games: K=3 keeps {n/(n+3):.0%} | K={K_USE:.0f} keeps {n/(n+K_USE):.0%}")

In [ ]:
# ---------- how do the players you picked change? ----------
cmp_rows = []
for lg, p in rated.items():
    a = rc.player_table(p, dict(cfg, K=3))
    b = rc.player_table(p, dict(cfg, K=K_USE))
    m = a[["playername", "position", "team", "games", "overall_final", "rank_in_role"]].merge(
        b[["playername", "position", "overall_final", "rank_in_role"]], on=["playername", "position"],
        suffixes=("_old", "_new"))
    m.insert(0, "league", lg)
    cmp_rows.append(m)
cmp = pd.concat(cmp_rows, ignore_index=True)
names = {"overall_final_old": "rating (old K=3)", "rank_in_role_old": "rank (old K=3)",
         "overall_final_new": f"rating (new K={K_USE:.1f})", "rank_in_role_new": f"rank (new K={K_USE:.1f})"}

look = cmp[cmp["playername"].isin(LOOK_AT)].rename(columns=names)
print(look.round(2).to_string(index=False) if len(look) else f"None of {LOOK_AT} found among players with at least {MIN_GAMES} games.")

cmp["rank_change"] = (cmp["rank_in_role_new"].astype(float) - cmp["rank_in_role_old"].astype(float)).abs()
moved = cmp.sort_values("rank_change", ascending=False).head(10).rename(columns=names)
print("\nBiggest rank changes in the league's role list when shrinkage is stronger:")
print(moved[["league", "playername", "position", "games", "rank (old K=3)", f"rank (new K={K_USE:.1f})"]].to_string(index=False))

In [ ]:
# ---------- 2) per-league baseline vs one pooled baseline ----------
pooled = rc.rate(pd.concat(raw.values(), ignore_index=True), cfg)      # one baseline across all chosen leagues

print("Average pooled game rating by league and role (1.00 = average of ALL chosen leagues together).")
print("If a league sits well above or below 1.00, region style / strength shifts the ratings.\n")
print(pooled.pivot_table(index="league", columns="position", values="r_overall", aggfunc="mean")[["top", "jng", "mid", "bot", "sup"]].round(3).to_string())

# player-level comparison
key = ["league", "playername", "position"]
own = pd.concat([p.assign(league=lg) for lg, p in rated.items()]).groupby(key)["r_overall"].agg(own_baseline="mean", games="size")
pld = pooled.groupby(key)["r_overall"].mean().rename("pooled_baseline")
both = own.join(pld).reset_index()
both = both[both["games"] >= MIN_GAMES]

print("\nRank correlation between the two versions, inside each league:")
for lg, g in both.groupby("league"):
    print(f"  {lg:5s} {g['own_baseline'].rank().corr(g['pooled_baseline'].rank()):.2f}   ({len(g)} players)")

top = both.sort_values("pooled_baseline", ascending=False).head(30)
share = both["league"].value_counts(normalize=True)
print("\nWhere the top 30 players come from under the pooled baseline, versus each league's share of all players:")
print(pd.DataFrame({"in top 30": top["league"].value_counts(), "share of players": (share * 30).round(1)}).fillna(0).to_string())

## How to read this

* **K estimate:** if most leagues give K around 8 to 20, use that for the site. A much higher K for one league (probably LPL) means its ratings are noisier, so players there should be pulled harder toward 1.00.
* **Pooled table:** league averages within roughly 0.97 to 1.03 mean the baseline choice hardly matters. A league far from 1.00 means region style leaks into the ratings, and a league-strength adjustment is needed before one global leaderboard is fair.
* **Top 30 by region:** if one region takes far more spots than its share of players, either its players are better, or style inflates its ratings. Decide by looking at the names.